In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def generate_contiguous_grid(n):
    """
    Generates an n x n board with contiguous groups of the same number,
    ensuring that numbers are always directly adjacent (up, down, left, right).
    Ensures that no region has more than n*n/6 cells and there are at most n regions.
    """
    board = np.zeros((n, n), dtype=int)
    current_number = 1
    regions = {}
    max_region_size = n * n // 6  # Define the maximum size of each region

    for row in range(n):
        for col in range(n):
            if row == 0 and col == 0:
                # Start with the first number
                board[row][col] = current_number
            elif row == 0:
                # First row: ensure left adjacency
                if len(regions.get(board[row][col - 1], [])) < max_region_size:
                    board[row][col] = board[row][col - 1]
                else:
                    current_number += 1
                    if current_number > n:
                        current_number = 1
                    board[row][col] = current_number
            elif col == 0:
                # First column: ensure top adjacency
                if len(regions.get(board[row - 1][col], [])) < max_region_size:
                    board[row][col] = board[row - 1][col]
                else:
                    current_number += 1
                    if current_number > n:
                        current_number = 1
                    board[row][col] = current_number
            else:
                # General case: choose a number that matches either left or above
                if board[row - 1][col] == board[row][col - 1]:
                    if len(regions.get(board[row - 1][col], [])) < max_region_size:
                        board[row][col] = board[row - 1][col]
                    else:
                        current_number += 1
                        if current_number > n:
                            current_number = 1
                        board[row][col] = current_number
                else:
                    if len(regions.get(board[row - 1][col], [])) < max_region_size:
                        board[row][col] = board[row - 1][col]
                    elif len(regions.get(board[row][col - 1], [])) < max_region_size:
                        board[row][col] = board[row][col - 1]
                    else:
                        current_number += 1
                        if current_number > n:
                            current_number = 1
                        board[row][col] = current_number

            # Record the regions (numbers) in a dictionary for later queen placement
            if board[row][col] not in regions:
                regions[board[row][col]] = []
            regions[board[row][col]].append((row, col))

            # Occasionally, transition to a new number
            if np.random.rand() < 0.1 and row > 0 and col > 0:
                current_number += 1
                if current_number > n:
                    current_number = 1
                board[row][col] = current_number

            # Ensure that the number of regions does not exceed n
            if len(regions) > n:
                break

    # Ensure that all numbers up to the highest number appear in the dictionary
    for num in range(1, n + 1):
        if num not in regions:
            regions[num] = []  # Add empty region for numbers that were never used

    return board, regions

def print_board_with_queens(board, queens):
    """
    Prints the board with queens placed on it.
    Queens are marked with 'Q' in the output.
    """
    board_with_queens = board.astype(str)
    for row, col in queens:
        board_with_queens[row][col] = 'Q'
    
    for row in board_with_queens:
        print(" ".join(row))
    print()

def is_safe(queen, queens, rows, cols, main_diag, anti_diag):
    """
    Checks if placing a queen at a specific position is safe.
    Ensures no other queen is on the same row, column, or diagonal.
    Also ensures no queens are placed adjacent diagonally.
    """
    row, col = queen
    if row in rows or col in cols or (row - col) in main_diag or (row + col) in anti_diag:
        return False
    return True

def solve_n_queens(board, n, regions):
    """
    Solves the N-Queens problem with backtracking, showing progress.
    Places exactly one queen per region while ensuring no queens share row, column, or diagonal.
    """
    queens = []
    
    # Set to keep track of blocked rows, columns, and diagonals
    rows = set()
    cols = set()
    main_diag = set()  # For diagonals: row - col
    anti_diag = set()  # For diagonals: row + col
    
    # Shuffle the regions to ensure randomness in placement
    region_positions = list(regions.values())
    
    for positions in region_positions:
        placed = False
        np.random.shuffle(positions)  # Shuffle positions to try different placements
        
        for pos in positions:
            if is_safe(pos, queens, rows, cols, main_diag, anti_diag):
                # Place the queen in the current position
                queens.append(pos)
                row, col = pos
                rows.add(row)
                cols.add(col)
                main_diag.add(row - col)
                anti_diag.add(row + col)
                placed = True
                break  # Stop once a queen is placed in the region
        
        if not placed:
            return []  # If we can't place a queen in the region, return no solution
    
    return queens

def plot_colored_regions(board, queens):
    """
    Plots the board with colored regions and marks queens.
    """
    n = board.shape[0]
    unique_numbers = np.unique(board)
    colors = list(mcolors.TABLEAU_COLORS.values())
    color_map = {num: colors[i % len(colors)] for i, num in enumerate(unique_numbers)}

    fig, ax = plt.subplots()
    for row in range(n):
        for col in range(n):
            rect = plt.Rectangle([col, n-row-1], 1, 1, color=color_map[board[row, col]])
            ax.add_patch(rect)
            if (row, col) in queens:
                ax.text(col + 0.5, n - row - 1 + 0.5, 'Q', ha='center', va='center', color='white', fontsize=12, fontweight='bold')
            else:
                ax.text(col + 0.5, n - row - 1 + 0.5, str(board[row, col]), ha='center', va='center', color='black')

    plt.xlim(0, n)
    plt.ylim(0, n)
    plt.gca().invert_yaxis()
    plt.axis('off')
    plt.show()

# Define board size
N = 8

# Retry logic until a valid grid with a solution is found
while True:
    board, regions = generate_contiguous_grid(N)
    queens = solve_n_queens(board, N, regions)
    
    if queens:
        # If a valid solution is found, print the result
        print("Generated Board:")
        print(board)
        print("\nBoard with Queens:")
        print_board_with_queens(board, queens)
        plot_colored_regions(board, queens)
        break  # Exit the loop once a valid solution is found